# JupyterLite で学ぶ itables 入門チュートリアル（DataFrame を対話的に表示する）

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、pandas の DataFrame を
**検索・並べ替え・ページ送りができる対話的な表** として表示するライブラリ **itables** を学ぶためのチュートリアルです。

## 対象者
- pandas の DataFrame を扱ったことがある方
- 行数の多い表を、スクロールや `head()` の繰り返しなしに眺めたい方
- 分析結果を見やすく共有したい方

## このチュートリアルで学ぶこと
0. 環境準備（JupyterLite 用）
1. itables とは
2. 練習用データの準備
3. `show()` の基本と大きな表の扱い
4. `all_interactive=True` ですべての DataFrame を対話表示にする
5. `show()` のオプション（検索、並べ替え、ページ長、列の非表示、見た目）
6. 数値の書式
7. pandas の `style` との使い分け
8. 集計結果（groupby / pivot）の表示
9. まとめと総合演習

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。
- 表の右上の検索ボックスに文字を入れる、列見出しをクリックして並べ替える、下のページ番号を押す、といった操作を実際に試してみてください。

---
## 0. 環境準備（JupyterLite 用）

`itables` は純 Python のライブラリなので、`piplite` で PyPI からインストールできます。
表示には JavaScript ライブラリ **DataTables** を使います（`init_notebook_mode()` がノートブックに組み込みます）。

In [ ]:
# JupyterLite 用のパッケージインストール（ローカルの Jupyter ではスキップされます）
try:
    import piplite
    await piplite.install(["matplotlib", "jinja2", "itables", "pandas", "numpy"])
except ImportError:
    pass

import jinja2      # pandas の DataFrame.style（Styler）が内部で使うため先に読み込む
import matplotlib  # background_gradient などの色付けで使う


In [ ]:
import numpy as np
import pandas as pd
import itables
from itables import init_notebook_mode, show

# 対話表示の準備。all_interactive=False：show() を使ったときだけ対話表示にする
init_notebook_mode(all_interactive=False)

print("itables バージョン:", itables.__version__)
print("pandas バージョン :", pd.__version__)

---
## 1. itables とは

Jupyter で `df` と書いて表示される表は、先頭と末尾の数行しか見えず、並べ替えや検索もできません。
**itables** は、DataFrame を JavaScript の **DataTables** で描画し、次の操作をブラウザ上でできるようにします。

| できること | 操作 |
|---|---|
| 検索（絞り込み） | 右上の検索ボックスに文字を入力 |
| 並べ替え | 列見出しをクリック（もう一度で逆順） |
| ページ送り | 下のページ番号、1 ページの行数の変更 |
| スクロール表示 | 縦横のスクロール |

使い方は 2 通りあります。

- `show(df)`：この表だけ対話表示にする（`init_notebook_mode(all_interactive=False)` のとき）
- `init_notebook_mode(all_interactive=True)`：以後、すべての DataFrame が自動的に対話表示になる

> **JupyterLite での注意**: `init_notebook_mode()` は既定で DataTables の JavaScript をノートブック内に埋め込みます
> （`connected=True` を指定すると CDN から読み込みます）。表示されないときは、このセルを実行し直してください。

---
## 2. 練習用データの準備

架空の企業データ（1,000 社）を作ります。列は「都道府県・業種・設立年・売上（百万円）・従業員数・利益率」です。

In [ ]:
rng = np.random.default_rng(0)
n = 1000

prefectures = ["愛知県", "岐阜県", "三重県", "静岡県", "東京都", "大阪府", "福岡県", "北海道"]
industries = ["製造業", "小売業", "情報通信業", "建設業", "運輸業", "サービス業"]

firms = pd.DataFrame({
    "企業ID": [f"C{i:04d}" for i in range(1, n + 1)],
    "都道府県": rng.choice(prefectures, n, p=[0.25, 0.1, 0.1, 0.1, 0.2, 0.1, 0.1, 0.05]),
    "業種": rng.choice(industries, n),
    "設立年": rng.integers(1950, 2024, n),
    "売上_百万円": np.round(rng.lognormal(mean=6, sigma=1, size=n)).astype(int),
    "従業員数": rng.integers(5, 800, n),
})
firms["利益率"] = np.round(rng.normal(0.05, 0.04, n), 3)

print(firms.shape)
firms.head()   # 通常の表示（先頭 5 行だけ）

---
## 3. `show()` の基本と大きな表の扱い

`show(df)` で対話的な表になります。まずは先頭 20 行で試してみましょう。
右上の検索ボックスに「製造業」と入力したり、「売上_百万円」の見出しをクリックしたりしてみてください。

In [ ]:
show(firms.head(20))

### 3.1 大きな表：自動的な間引き（downsampling）

1,000 行の表をそのまま `show()` すると、itables は既定でデータ量を **64 KB 以下に間引いて** 表示します
（表の下に「downsampled」という注意書きが出ます）。ノートブックの容量が大きくなりすぎないための仕組みです。

- 全行を表示したいとき：`maxBytes=0`（無制限）
- 上限を変えたいとき：`maxBytes="200KB"` のように指定

In [ ]:
# 1,000 行を全部表示する（maxBytes=0 で間引きをやめる）
show(firms, maxBytes=0)

### 3.2 インデックスの表示

既定では、インデックスが「0, 1, 2, ...」の連番のときは表示されず、意味のあるインデックスのときは表示されます。
`showIndex=True` / `False` で明示的に切り替えられます。

In [ ]:
top10 = firms.sort_values("売上_百万円", ascending=False).head(10)
show(top10, showIndex=False)

### 練習問題 1

1. `firms` から「東京都」の企業だけを取り出し、`show()` で対話表示してください（インデックスは非表示）。
2. 従業員数の多い順に並べた上位 30 社を `show()` で表示してください。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
# 1
show(firms[firms["都道府県"] == "東京都"], showIndex=False)

# 2
show(firms.sort_values("従業員数", ascending=False).head(30), showIndex=False)
```

</details>

---
## 4. `all_interactive=True` ですべての DataFrame を対話表示にする

`init_notebook_mode(all_interactive=True)` を実行すると、以後は `show()` を書かなくても、
セルの最後に書いた DataFrame がすべて対話的な表になります。

In [ ]:
init_notebook_mode(all_interactive=True)

firms.head(50)    # show() なしでも対話表示になる

In [ ]:
# 集計結果も同じ
firms.groupby("業種")["売上_百万円"].describe().round(1)

元に戻すには `all_interactive=False` にします。このノートブックでは、以降は `show()` を明示的に使います。

In [ ]:
init_notebook_mode(all_interactive=False)
firms.head(3)     # 通常の表示に戻る

---
## 5. `show()` のオプション

`show()` には DataTables のオプションをキーワード引数として渡せます。よく使うものをまとめます。

| オプション | 意味 | 例 |
|---|---|---|
| `pageLength` | 1 ページの行数 | `pageLength=5` |
| `lengthMenu` | 行数の選択肢 | `lengthMenu=[5, 10, 25, 100]` |
| `order` | 初期の並べ替え（列番号, "asc"/"desc"） | `order=[[4, "desc"]]` |
| `layout` | 検索ボックスなどの配置。`{"topEnd": None}` で検索ボックスを消す | `layout={"topEnd": None}` |
| `paging` | ページ送りの有無 | `paging=False` |
| `scrollY` | 縦スクロールの高さ | `scrollY="300px"` |
| `columnDefs` | 列ごとの設定（非表示、幅、書式） | `columnDefs=[{"visible": False, "targets": [0]}]` |
| `classes` | 表の見た目（DataTables のクラス） | `classes="display compact cell-border"` |
| `caption` | 表のタイトル | `caption="企業一覧"` |

In [ ]:
# 1 ページ 5 行、行数の選択肢を指定、売上の降順で初期表示
show(firms.head(100), pageLength=5, lengthMenu=[5, 10, 25], order=[[4, "desc"]], caption="売上上位（100 社中）")

In [ ]:
# 検索とページ送りを消して、縦スクロールにする
show(firms.head(200), layout={"topEnd": None}, paging=False, scrollY="250px", showIndex=False)

In [ ]:
# 列を非表示にする（targets は 0 始まりの列番号。showIndex=False のとき 0 列目は「企業ID」）
show(firms.head(50), columnDefs=[{"visible": False, "targets": [0]}], showIndex=False, pageLength=10)

In [ ]:
# 見た目を変える（compact: 行間を詰める、cell-border: 罫線）と列幅の指定
show(firms.head(30), classes="display compact cell-border", columnDefs=[{"width": "140px", "targets": [1]}], showIndex=False, pageLength=10)

### 練習問題 2

1. `firms` の先頭 100 行を、1 ページ 10 行・利益率の降順で表示してください（利益率は 6 列目 → `targets` は 6）。
2. 「企業ID」と「設立年」の列を非表示にし、検索ボックスを消して（`layout={"topEnd": None}`）表示してください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
# 1
show(firms.head(100), pageLength=10, order=[[6, "desc"]], showIndex=False)

# 2
show(firms.head(100), columnDefs=[{"visible": False, "targets": [0, 3]}], layout={"topEnd": None}, showIndex=False)
```

</details>

---
## 6. 数値の書式

### 6.1 pandas 側で整える（簡単）

表示前に pandas で丸めておくのが最も簡単です。ただし、文字列に変換すると並べ替えが文字列順になってしまうので、
`round()` で数値のまま丸めるのがおすすめです。

In [ ]:
formatted = firms.head(30).copy()
formatted["利益率"] = (formatted["利益率"] * 100).round(1)          # % 表示用に数値のまま変換
formatted = formatted.rename(columns={"利益率": "利益率_%"})
show(formatted, showIndex=False, pageLength=10)

### 6.2 DataTables の書式機能を使う（発展）

`JavascriptCode` を使うと、DataTables 側の数値フォーマット（3 桁区切り、通貨記号など）を列に適用できます。
値は数値のままなので、並べ替えも正しく行われます。

`DataTable.render.number(千区切り, 小数点, 小数桁数, 接頭辞, 接尾辞)` の形です。

In [ ]:
from itables import JavascriptCode

show(
    firms.head(30),
    columnDefs=[
        {"targets": [4], "render": JavascriptCode("DataTable.render.number(',', '.', 0, '', ' 百万円')")},
        {"targets": [5], "render": JavascriptCode("DataTable.render.number(',', '.', 0, '', ' 人')")},
    ],
    showIndex=False,
    pageLength=10,
)

---
## 7. pandas の `style` との使い分け

pandas にも `df.style` という表示機能があり、条件付き書式（色付け）ができますが、**静的な HTML** なので検索や並べ替えはできません。

| | itables（`show`） | pandas `style` |
|---|---|---|
| 検索・並べ替え・ページ送り | できる | できない |
| 色付け・棒グラフ風の表示 | 基本的にできない | できる（`background_gradient`, `bar` など） |
| 向いている場面 | データを **探索** する | 結果を **見せる**（レポート） |

In [ ]:
# pandas の style：売上に色のグラデーション、利益率に棒
firms.head(10).style.background_gradient(subset=["売上_百万円"], cmap="Blues").bar(subset=["利益率"], color="#f4a261").format({"利益率": "{:.1%}"})

---
## 8. 集計結果（groupby / pivot）の表示

集計した表も `show()` で対話表示できます。`reset_index()` でインデックスを列に戻しておくと、検索・並べ替えの対象になります。

In [ ]:
by_pref_ind = (
    firms.groupby(["都道府県", "業種"])
    .agg(企業数=("企業ID", "count"), 平均売上=("売上_百万円", "mean"), 平均利益率=("利益率", "mean"))
    .round({"平均売上": 1, "平均利益率": 3})
    .reset_index()
)
show(by_pref_ind, pageLength=10, order=[[2, "desc"]], showIndex=False)

In [ ]:
pivot = firms.pivot_table(index="都道府県", columns="業種", values="売上_百万円", aggfunc="median").round(0)
show(pivot, paging=False, layout={"topEnd": None})

### 練習問題 3

1. 業種ごとの企業数・従業員数の合計・売上の中央値を求め、企業数の多い順に `show()` で表示してください。
2. 都道府県 × 設立年代（10 年ごと: `firms["設立年"] // 10 * 10`）の企業数の pivot 表を作り、`show()` で表示してください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
by_ind = (
    firms.groupby("業種")
    .agg(企業数=("企業ID", "count"), 従業員数合計=("従業員数", "sum"), 売上中央値=("売上_百万円", "median"))
    .reset_index()
)
show(by_ind, order=[[1, "desc"]], showIndex=False, paging=False)

# 2
decade = firms.assign(設立年代=firms["設立年"] // 10 * 10)
pv = decade.pivot_table(index="都道府県", columns="設立年代", values="企業ID", aggfunc="count", fill_value=0)
show(pv, paging=False, layout={"topEnd": None})
```

</details>

---
## 9. まとめ

| やりたいこと | 書き方 |
|---|---|
| 準備 | `from itables import init_notebook_mode, show`; `init_notebook_mode(all_interactive=False)` |
| 1 つの表を対話表示 | `show(df)` |
| すべての表を対話表示 | `init_notebook_mode(all_interactive=True)` |
| 全行を表示 | `show(df, maxBytes=0)` |
| インデックスを隠す | `show(df, showIndex=False)` |
| ページ長・並べ替え | `pageLength=10`, `lengthMenu=[...]`, `order=[[列番号, "desc"]]` |
| 検索・ページ送りを消す | `layout={"topEnd": None}`, `paging=False`, `scrollY="300px"` |
| 列を隠す・幅 | `columnDefs=[{"visible": False, "targets": [0]}]`, `{"width": "120px", "targets": [1]}` |
| 数値の書式 | pandas で `round()`、または `JavascriptCode("DataTable.render.number(...)")` |
| 見た目 | `classes="display compact cell-border"`, `caption="..."` |

## 次のステップ

- `python/pandas/pandas_intermediate_tutorial.ipynb` — groupby・pivot・結合で「見せたい表」を作る
- `python/openpyxl/openpyxl_xlsxwriter_beginner_tutorial.ipynb` — 表を Excel レポートとして書き出す
- `python/altair/altair_beginner_tutorial.ipynb` — 表ではなく対話的なグラフで見せる

---
## 総合演習：企業データの探索ビューを作る

`firms` を使って、次の 3 つの表を `show()` で作ってください。

1. **探索用の全件表**：全 1,000 行を、インデックス非表示・1 ページ 25 行・売上の降順で表示し、売上と従業員数に 3 桁区切りを付ける。
2. **都道府県別サマリー**：都道府県ごとの企業数・売上合計・平均利益率（小数第 3 位まで）を、売上合計の降順で、ページ送りなしで表示する。
3. **業種 × 都道府県の企業数**：pivot 表（欠損は 0）を、検索・ページ送りなしで表示する。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください

### 総合演習の解答例

In [ ]:
# 1. 探索用の全件表
show(
    firms,
    maxBytes=0,
    showIndex=False,
    pageLength=25,
    order=[[4, "desc"]],
    columnDefs=[{"targets": [4, 5], "render": JavascriptCode("DataTable.render.number(',', '.', 0)")}],
    caption="企業一覧（全 1,000 社）",
)

# 2. 都道府県別サマリー
by_pref = (
    firms.groupby("都道府県")
    .agg(企業数=("企業ID", "count"), 売上合計=("売上_百万円", "sum"), 平均利益率=("利益率", "mean"))
    .round({"平均利益率": 3})
    .reset_index()
)
show(by_pref, order=[[2, "desc"]], paging=False, showIndex=False, caption="都道府県別サマリー")

# 3. 業種 × 都道府県の企業数
pv = firms.pivot_table(index="業種", columns="都道府県", values="企業ID", aggfunc="count", fill_value=0)
show(pv, paging=False, layout={"topEnd": None}, caption="業種 × 都道府県の企業数")

お疲れさまでした！ itables を使うと、`head()` を繰り返さずにデータ全体を素早く眺められます。
探索には itables、レポートには pandas の `style` や Excel 出力、と使い分けてみてください。